# 03 — Eval

Phase 3 (Per-Field-Accuracy Baseline), Phase 4 (Iterations-Auswertung + Synthese-Tabelle), Phase 6 (Skalierung 7B + 3B-Halluzinations-Klassen).

Predictions kommen aus `02_extract.ipynb`. Gold ist eure `annotation/meine_gold.csv` aus Phase 2.

## Run-Header

| Feld | Wert |
|---|---|
| Datum | _YYYY-MM-DD_ |
| Gold-Datei | `annotation/meine_gold.csv` |
| Aktive Predictions-Datei(en) | _ |
| Match-Entscheidung `skills_top3` | _ (Set / geordnete Liste) |
| Match-Entscheidung `gehalt_min_eur` | _ (exakt / Toleranz X %) |
| JSON-Parse-Fails (Anzahl) | _ |

## Phase 3 — Baseline-Accuracy auf 12 Hand-Gold-Anzeigen

Hypothese-Cell *vor* der Eval: welches Feld haltet ihr für am stärksten / am schwächsten — und warum?

### Hypothese vor der Eval

Basierend auf dem κ-Befund aus Phase 2 und der Korpus-Inspektion:

- **Erwartet am stärksten:** `vertragsart` — die API liefert mit `stellenangebotsart` eine starke Vorinformation; selbst über den reinen Text sind Ausbildung / Festanstellung / Praktikum meist eindeutig. κ Mensch↔Mensch war hier 0.826.
- **Erwartet am schwächsten:** `homeoffice` — schon zwischen Menschen κ=0.122. Ambivalente Formulierungen wie *„Möglichkeit zum hybriden Arbeiten"*, *„Gleitzeit und Homeoffice"*, *„nach Absprache"* werden auch das Modell schwanken lassen, vor allem zwischen `ja` / `teilweise` / `nicht_genannt`.
- **Wildcard `gehalt_min_eur`:** in unserem Korpus haben ~80% gar keine €-Zahl im Text → viele Gold-`null`s → Feld wirkt künstlich „leicht", solange das Modell brav `null` schreibt. Echtes Signal kommt aus den 2–3 Anzeigen mit konkreter Zahl (z. B. Ausbildung mit „1.100 € monatlich").

**Match-Entscheidungen** (vorab dokumentieren, gehören in Stufe-3-Bewertung):
- `skills_top3`: **Set-Match** (case-insensitive). Reihenfolge ist subjektiv, ein Modell-Output `["SQL", "Python"]` und Gold `["Python", "SQL"]` zählen als Match.
- `gehalt_min_eur`: **Toleranz ±5%**. Eine Anzeige mit Range „50.000–60.000 €" kann je nach Lesart 50000 (Untergrenze) oder 55000 (Mitte) liefern; 5% deckt das ab, ohne grobe Halluzinationen zu kaschieren. `null` muss `null` matchen (kein Toleranz-Spielraum).
- `_parse_fail`: zählt für **alle 6 Felder** dieser Anzeige als Miss.

In [ ]:
# ── Setup: Imports + Gold laden + Match-Funktionen + Baseline-Eval ─────────
import json
import pandas as pd
from pathlib import Path

GOLD_PATH      = Path("../annotation/meine_gold.csv")
PRED_PATH_BASE = Path("predictions.jsonl")

# Gold laden + normalisieren (lower-case, leere → None, skills als Set)
gold_df = pd.read_csv(GOLD_PATH)
gold_df = gold_df.dropna(subset=["id"])

def _nstr(v):
    if pd.isna(v): return None
    s = str(v).strip()
    return None if s.lower() in ("", "null", "nan") else s.lower()

def _nint(v):
    if pd.isna(v): return None
    s = str(v).strip()
    if s.lower() in ("", "null", "nan"): return None
    try: return int(float(s))
    except (ValueError, TypeError): return None

def _nskills(v):
    if pd.isna(v): return set()
    s = str(v).strip()
    if s.lower() in ("", "null", "nan", "nicht_genannt"): return set()
    return {x.strip().lower() for x in s.split("|")
            if x.strip() and x.strip().lower() != "nicht_genannt"}

gold_by_id = {
    str(r["id"]): {
        "homeoffice":      _nstr(r["homeoffice"]),
        "vertragsart":     _nstr(r["vertragsart"]),
        "erfahrungslevel": _nstr(r["erfahrungslevel"]),
        "gehalt_min_eur":  _nint(r["gehalt_min_eur"]),
        "gehalt_zeitraum": _nstr(r["gehalt_zeitraum"]),
        "skills_top3":     _nskills(r["skills_top3"]),
    }
    for _, r in gold_df.iterrows()
}
print(f"Gold geladen: {len(gold_by_id)} Anzeigen")

# Match-Funktionen pro Feld (Schema-Match-Entscheidungen aus Hypothese-Cell)
def _eq(a, b):
    if a is None and b is None: return True
    if a is None or b is None: return False
    return str(a).strip().lower() == str(b).strip().lower()

def _eq_gehalt(a, b):  # Toleranz ±5%, null muss null matchen
    if a is None and b is None: return True
    if a is None or b is None: return False
    try:
        a, b = int(a), int(b)
        return a == 0 if b == 0 else abs(a - b) / abs(b) <= 0.05
    except (TypeError, ValueError):
        return False

def _eq_skills(pred, gold):  # Set-Match case-insensitive
    if isinstance(pred, str): pred = [pred]
    if pred is None: pred = []
    pred_set = {str(x).strip().lower() for x in pred
                if x and str(x).strip().lower() != "nicht_genannt"}
    gold_set = gold if isinstance(gold, set) else set()
    return pred_set == gold_set

FELDER = [
    ("homeoffice",      _eq),
    ("vertragsart",     _eq),
    ("erfahrungslevel", _eq),
    ("gehalt_min_eur",  _eq_gehalt),
    ("gehalt_zeitraum", _eq),
    ("skills_top3",     _eq_skills),
]

# eval_run: lädt predictions.jsonl, rechnet Per-Field-Accuracy gegen gold_by_id
def eval_run(pred_path, label):
    preds = {}
    for line in pred_path.open(encoding="utf-8"):
        obj = json.loads(line)
        preds[obj["refnr"]] = obj
    common = sorted(set(preds) & set(gold_by_id))
    fails  = sum(1 for r in common if preds[r].get("_parse_fail"))
    rows, miss = [], []
    for feld, matcher in FELDER:
        n_ok = 0
        for rid in common:
            p, g = preds[rid], gold_by_id[rid]
            ok = False if p.get("_parse_fail") else matcher(p.get(feld), g.get(feld))
            if ok: n_ok += 1
            else:  miss.append({"run": label, "refnr": rid, "feld": feld,
                                "pred": p.get(feld), "gold": g.get(feld),
                                "parse_fail": bool(p.get("_parse_fail"))})
        rows.append({"feld": feld, f"n_{label}": n_ok,
                     f"acc_{label}": n_ok / len(common)})
    return pd.DataFrame(rows), fails, miss

# Baseline-Run laden + auswerten
acc_df, parse_fails, misses = eval_run(PRED_PATH_BASE, "korrekt")
acc_df = acc_df.rename(columns={"acc_korrekt": "accuracy"})

print(f"\nBaseline-Accuracy (Parse-Fails: {parse_fails})")
print(acc_df.to_string(index=False))
print(f"\nGesamt-Accuracy (Mittel über 6 Felder): {acc_df['accuracy'].mean():.1%}")

## Phase 4 — Iteration A Auswertung

**Hypothese vor Iteration A:** Schärferer Prompt mit präzisen Homeoffice-Regeln (inkl. Edge-Case-Mapping aus Phase 2) hebt `homeoffice` um +2–4 Treffer. Wenn das funktioniert → Diagnose **Schema-Problem** (Schema war zu vage), nicht reines Modell-Problem.

In [ ]:
def eval_run(pred_path: Path, label: str) -> tuple[pd.DataFrame, int, list[dict]]:
    """Lädt eine predictions.jsonl, rechnet Per-Field-Accuracy gegen gold_by_id, gibt (df, parse_fails, miss_details) zurück."""
    preds = {}
    for line in pred_path.open(encoding="utf-8"):
        obj = json.loads(line)
        preds[obj["refnr"]] = obj
    common_ids = sorted(set(preds) & set(gold_by_id))
    fails = sum(1 for r in common_ids if preds[r].get("_parse_fail"))

    rows_, misses_ = [], []
    for feld, matcher in FELDER:
        n_match = 0
        for rid in common_ids:
            p, g = preds[rid], gold_by_id[rid]
            ok = False if p.get("_parse_fail") else matcher(p.get(feld), g.get(feld))
            if ok:
                n_match += 1
            else:
                misses_.append({"run": label, "refnr": rid, "feld": feld,
                                "pred": p.get(feld), "gold": g.get(feld),
                                "parse_fail": bool(p.get("_parse_fail"))})
        rows_.append({"feld": feld, f"n_{label}": n_match,
                      f"acc_{label}": n_match / len(common_ids)})
    return pd.DataFrame(rows_), fails, misses_


# Iteration A laden + evaluieren
PRED_PATH_A = Path("predictions_iter_A.jsonl")
iter_A_df, parse_fails_A, misses_A = eval_run(PRED_PATH_A, "A")

# Δ gegen Baseline
delta_A = acc_df[["feld", "n_korrekt", "accuracy"]].rename(
    columns={"n_korrekt": "n_baseline", "accuracy": "acc_baseline"}
).merge(iter_A_df, on="feld")
delta_A["Δ_korrekt"] = delta_A["n_A"] - delta_A["n_baseline"]
delta_A["Δ_pt"]      = ((delta_A["acc_A"] - delta_A["acc_baseline"]) * 100).round(0).astype(int)

print(f"Iteration A vs. Baseline  (Parse-Fails: baseline={parse_fails}, A={parse_fails_A})\n")
print(delta_A[["feld", "n_baseline", "n_A", "Δ_korrekt", "Δ_pt"]].to_string(index=False))

# Misses bei homeoffice (Hauptziel von A) im Detail
print("\nHomeoffice-Misses unter Iteration A:")
for m in [x for x in misses_A if x["feld"] == "homeoffice"]:
    tag = " [PARSE_FAIL]" if m["parse_fail"] else ""
    print(f"  {m['refnr']}{tag}: pred={m['pred']!r:25s}  gold={m['gold']!r}")

## Phase 4 — Iteration B Auswertung

**Hypothese vor Iteration B:** Few-Shot-Beispiele aus dem Hand-Gold stabilisieren primär `skills_top3` (Format/Filter besser) und `gehalt_min_eur`/`gehalt_zeitraum`-Konsistenz. `homeoffice`-Ambivalenz löst sich aber nicht durch Beispiele auf — dort erwarte ich keinen großen Sprung gegen Baseline. Wenn doch, ist's eher das **Format-Lernen** als die Schema-Klärung.

In [ ]:
# Iteration B laden + evaluieren (gleiche Mechanik wie für A)
PRED_PATH_B = Path("predictions_iter_B.jsonl")
iter_B_df, parse_fails_B, misses_B = eval_run(PRED_PATH_B, "B")

delta_B = acc_df[["feld", "n_korrekt", "accuracy"]].rename(
    columns={"n_korrekt": "n_baseline", "accuracy": "acc_baseline"}
).merge(iter_B_df, on="feld")
delta_B["Δ_korrekt"] = delta_B["n_B"] - delta_B["n_baseline"]
delta_B["Δ_pt"]      = ((delta_B["acc_B"] - delta_B["acc_baseline"]) * 100).round(0).astype(int)

print(f"Iteration B vs. Baseline  (Parse-Fails: baseline={parse_fails}, B={parse_fails_B})\n")
print(delta_B[["feld", "n_baseline", "n_B", "Δ_korrekt", "Δ_pt"]].to_string(index=False))

# skills_top3-Misses unter Iteration B (Hauptziel)
print("\nSkills_top3-Misses unter Iteration B:")
for m in [x for x in misses_B if x["feld"] == "skills_top3"]:
    tag = " [PARSE_FAIL]" if m["parse_fail"] else ""
    print(f"  {m['refnr']}{tag}: pred={m['pred']!r:50s}  gold={m['gold']!r}")

## Phase 4 — Iteration A Auswertung

Hypothese-Cell *vor* der Iteration: welches Feld, welche Δ-Größe, warum?

In [ ]:
# Iterations-Tabelle: Baseline | A | B nebeneinander
synth = (
    acc_df[["feld", "n_korrekt"]].rename(columns={"n_korrekt": "Baseline"})
    .merge(iter_A_df[["feld", "n_A"]].rename(columns={"n_A": "Iter A"}), on="feld")
    .merge(iter_B_df[["feld", "n_B"]].rename(columns={"n_B": "Iter B"}), on="feld")
)
synth["Δ A"] = synth["Iter A"] - synth["Baseline"]
synth["Δ B"] = synth["Iter B"] - synth["Baseline"]

print("Iterations-Tabelle (n_korrekt von 12 pro Feld):\n")
print(synth.to_string(index=False))

# Gesamt-Accuracy je Run (Mittel über alle 6 Felder)
print(f"\nGesamt-Accuracy (Mittel über 6 Felder):")
print(f"  Baseline : {acc_df['accuracy'].mean():.1%}")
print(f"  Iter A   : {iter_A_df['acc_A'].mean():.1%}")
print(f"  Iter B   : {iter_B_df['acc_B'].mean():.1%}")
print(f"\nParse-Fails: baseline={parse_fails}, A={parse_fails_A}, B={parse_fails_B}")

# Statistische Selbstkritik (n=12: ~8.3 Pt pro Anzeige)
print(f"\nSignal-vs-Rausch-Schwelle bei n=12: 1 Anzeige = {100/12:.1f} Pt.")
print("Befunde ab |Δ| ≥ 2 Anzeigen (≈ 17 Pt) ernst nehmen — alles darunter ist im Rauschen.")

### Synthese

| Iteration | Hypothese | Aktion | Δ Gesamt | Δ schwächstes Feld | Diagnose |
|---|---|---|---|---|---|
| **Baseline** | — | — | 52.8% | `skills_top3` 0/12, `erfahrungslevel` 3/12 | — |
| **A** | Schärferer Prompt mit präzisen Homeoffice-Regeln hebt `homeoffice` um +2–4 Treffer → wäre Beleg für **Schema-Problem** | Prompt-Klarstellung (Homeoffice-Regeln + Edge-Case-Mapping + Vertragsart-Regeln) | +6.9 pt (59.7%) | `homeoffice` nur +1 (8 pt, **im Rauschen**); `erfahrungslevel` +2 (17 pt, signifikant); `skills_top3` +2 (17 pt) | **Hypothese widerlegt für homeoffice** — schärferer Prompt allein reicht nicht. 7B beachtet abstrakte Regeln im System-Prompt zu wenig. → eher **Modell-Limit** als reines Schema-Problem |
| **B** | Few-Shots stabilisieren primär **`skills_top3`-Format**, `homeoffice` erwarte ich kaum Veränderung | 3 Few-Shot-Paare (Ausbildung, Senior-Festanstellung, Werkstudent) vor User-Turn | **+15.3 pt (68.1%)** | `homeoffice` **+5 (42 pt, sehr signifikant)**; `erfahrungslevel` +3 (25 pt); `skills_top3` +2 (17 pt) | **Hypothese ebenfalls widerlegt** — Few-Shots halfen am stärksten bei den **kategorischen Feldern** (`homeoffice`, `erfahrungslevel`), nicht primär beim Format. Konkrete Demonstration der *Werte-Auswahl* schlägt abstrakte Regel-Beschreibung |

**Gemeinsame Befunde:**
- `gehalt_min_eur` + `gehalt_zeitraum`: 10/12 in allen drei Runs — vollständig stabil, weil ~80% der Anzeigen keine €-Zahl enthalten und Baseline brav `null` schreibt
- `vertragsart`: 11→12 (Iter B) — Plafond erreicht, durch Beispiel mit Werkstudent wurde der eine Miss aufgeholt

---

### Drei Synthese-Antworten (aus Aufgabenblatt Phase 4.4)

**1. Welcher der drei Fehler-Typen war in meinem Setup die häufigste Ursache?**

Hauptsächlich **Modell-Limit**, nicht Schema-Problem. Die Erwartung war, dass `homeoffice` ein Schema-Problem ist (schon κ zwischen Menschen war nur 0.122), und ein präziserer Prompt müsste es lösen. Iter A widerlegte das: trotz expliziter Regeln (`"hybrid" → teilweise`, `"nach Absprache" → teilweise`) und Edge-Case-Mapping blieb der Δ-Effekt im Rauschen (+1 von 12). Die Predictions zeigen: das 7B-Modell sagte 5× `nicht_genannt`, obwohl der Begriff im Text stand (z.B. "mobiles Arbeiten") — es hat die Regel im System-Prompt schlicht **nicht aktiv angewendet**.

Iter B löste dasselbe Feld mit Few-Shots fast doppelt so gut (+5). Diagnose: 7B kann **konkrete Vorbilder besser umsetzen als abstrakte Regeln**. Das ist ein Modell-Limit in der Abstraktions-/Instruction-Following-Fähigkeit, kein Schema-Limit.

Sekundär bei `skills_top3` ein **Matcher-/Schema-Problem**: Predictions wie `["Power BI", "SQL", "Python"]` matchen `{"power bi", "sql", "python"}` zwar — aber Hand-Annotationen sind subjektiv (`"MS Office"` vs `"Excel"`, `"PowerBI"` vs `"Power BI"`, ich annotierte `"sas"` während Modell `"SAS"` schreibt — nach lowercasing matcht das). Das **0/12** in der Baseline ist also irreführend: die Predictions sind oft *plausibel*, nur set-mäßig nicht identisch.

**2. Was wäre nicht durch Iteration lösbar — wo ist Modell oder Schema die harte Grenze?**

- **`skills_top3` ist inhärent subjektiv**: Schema sagt "max. 3 technische Skills", aber *welche* 3 aus 8-10 genannten? Beide Annotatoren (ich + Partner) zogen unterschiedliche Skills. Selbst perfekte Modell-Output kann durch Set-Match nie 100% erreichen. Lösung wäre **Schema-Änderung**: alle erwähnten Skills extrahieren + Canonicalization-Liste (`"PowerBI" → "Power BI"`), nicht "Top 3".
- **`gehalt_min_eur` bei Range-Anzeigen**: Schema sagt "Untergrenze nehmen", aber bei `"50.000-60.000 €"` ist auch `55000` (Mittelwert) eine plausible Lesart. ±5%-Toleranz im Matcher fängt das ab — ohne diese Pipeline-Entscheidung wäre Δ negativ messbar gewesen.
- **`homeoffice nicht_genannt` vs `teilweise`**: in 5 Fällen ist die Anzeige real ambivalent (sagt z.B. nur "Benefits: Homeoffice"). Selbst Iter B löst das nicht vollständig — auch die Few-Shots können die echte Ambiguität nicht auflösen, weil die Anzeigen *real* unterspezifiziert sind. **Schema-Limit**.

**3. Würde ich im Berufsalltag alle Probleme fixen, oder gibt es Fehler, die „gut genug" sind?**

- **`vertragsart` mit 100% (Iter B)**: perfekt — keine weitere Iteration nötig. Plus: API liefert `stellenangebotsart` als Strukturfeld, das wäre ein **Pipeline-Hebel** statt LLM (LLM braucht's hier gar nicht).
- **`homeoffice` mit 75% (Iter B)**: grenzwertig. Für **Reporting-Aggregate** ("wie viele Anzeigen bieten Homeoffice?") ist 75% OK — der Bias mittelt sich raus. Für **Recruiting-Filter** ("zeige mir nur Remote-Stellen") wäre 75% zu wenig, da kämen False Positives in der Trefferliste.
- **`skills_top3` mit 17%**: wenn das Reporting nur auf **Skill-Häufigkeit über alle Anzeigen** zielt, ist das Δ zwischen Modell-Output und Gold marginal (Set-Match versteckt es). Für **Pflicht-Skills pro Anzeige** ("hat die Stelle Python?") wäre fuzzy-Match auf einzelne Skills die bessere Metrik — die wäre vermutlich bei 70%+.
- **`gehalt_min_eur` mit 83%**: täuschend gut, weil ~80% der Anzeigen Gold-`null` haben. Die echten Trefferquoten auf den 2-3 Anzeigen mit konkreter Zahl interessieren mehr. Für eine **Gehalts-Aggregation** wäre das nicht ausreichend.

→ **"Gut genug" ist immer Use-Case-abhängig.** Im echten Projekt würde ich Iter B als Pipeline einfrieren, aber pro Feld einen Schwellwert + Review-Stichprobe definieren statt blind die Gesamt-68.1% zu kommunizieren.

In [ ]:
# ── Block 6.1: 7B-Run auf 12 Hand-Gold-Anzeigen evaluieren ────────────────
PRED_PATH_7B = Path("predictions_7b_full.jsonl")
preds_7b_all = {}
for line in PRED_PATH_7B.open(encoding="utf-8"):
    obj = json.loads(line)
    preds_7b_all[obj["refnr"]] = obj

print(f"7B-Full: {len(preds_7b_all)} Anzeigen total")

# Auf den 12 Gold: Accuracy berechnen + mit Phase-4-Baseline vergleichen
common_gold = sorted(set(preds_7b_all) & set(gold_by_id))
parse_fails_7b_gold = sum(1 for r in common_gold if preds_7b_all[r].get("_parse_fail"))

rows_7b = []
for feld, matcher in FELDER:
    n_match = sum(
        0 if preds_7b_all[r].get("_parse_fail") else int(matcher(preds_7b_all[r].get(feld), gold_by_id[r].get(feld)))
        for r in common_gold
    )
    rows_7b.append({"feld": feld, "n_7b_gold": n_match,
                    "acc_7b_gold": n_match / len(common_gold)})

print(f"\n7B-Full Accuracy auf den 12 Gold-Anzeigen (Stabilität vs. Phase 4):\n")
print(pd.DataFrame(rows_7b).to_string(index=False))
print(f"\nParse-Fails (7B auf Gold): {parse_fails_7b_gold}/{len(common_gold)}")
print("\nDanach Schema-Konformitätscheck auf den restlichen 78 (im Terminal):")
print("  python annotation/validate.py --validate-jsonl predictions_7b_full.jsonl")

In [ ]:
# ── Block 6.2: 3B-vs-7B-vs-Gold Vergleich, drei Halluzinations-Klassen ────
PRED_PATH_3B = Path("predictions_3b_full.jsonl")
preds_3b_all = {}
for line in PRED_PATH_3B.open(encoding="utf-8"):
    obj = json.loads(line)
    preds_3b_all[obj["refnr"]] = obj

print(f"3B-Full: {len(preds_3b_all)} Anzeigen total")

# Join: refnr → {7B-Wert, 3B-Wert, Gold-Wert} pro Feld
SCHEMA_FELDER = [f for f, _ in FELDER]
join_rows = []
for rid in sorted(set(preds_7b_all) & set(preds_3b_all)):
    p7, p3 = preds_7b_all[rid], preds_3b_all[rid]
    gold = gold_by_id.get(rid)  # None für die 78 ohne Hand-Gold
    for feld in SCHEMA_FELDER:
        join_rows.append({
            "refnr": rid, "feld": feld,
            "7b": p7.get(feld), "3b": p3.get(feld),
            "gold": (gold.get(feld) if gold else None),
            "has_gold": gold is not None,
            "7b_parse_fail": bool(p7.get("_parse_fail")),
            "3b_parse_fail": bool(p3.get("_parse_fail")),
        })
join = pd.DataFrame(join_rows)

# Wo weicht 3B von 7B ab?  ("vermutliche 3B-Halluzination" wenn 7B mit Gold matcht und 3B ≠ 7B)
def _eq(field, a, b):
    matcher = dict(FELDER)[field]
    return matcher(a, b)

mask_div = join.apply(lambda r: not _eq(r["feld"], r["7b"], r["3b"]), axis=1)
divergent = join[mask_div].copy()
mask_3b_hallu = divergent.apply(
    lambda r: r["has_gold"] and _eq(r["feld"], r["7b"], r["gold"]) and not _eq(r["feld"], r["3b"], r["gold"]),
    axis=1
)
divergent["3b_hallu_vs_gold"] = mask_3b_hallu

print(f"\n3B≠7B Abweichungen total: {len(divergent)}")
print(f"davon vermutliche 3B-Halluzinationen (7B==Gold, 3B≠Gold): {mask_3b_hallu.sum()}")
print(f"\nAbweichungen pro Feld:")
print(divergent.groupby("feld").size().sort_values(ascending=False).to_string())

# Top-Beispiele zeigen — als Material für die Halluzinations-Klassen
print("\nBeispiele für 3B-Abweichungen vs. 7B (sortiert nach Feld):\n")
for feld in SCHEMA_FELDER:
    sub = divergent[divergent["feld"] == feld].head(5)
    if sub.empty:
        continue
    print(f"── {feld} ({len(divergent[divergent['feld'] == feld])} Abweichungen) ──")
    for _, r in sub.iterrows():
        gold_tag = f" [GOLD: {r['gold']!r}]" if r["has_gold"] else ""
        hallu_tag = " ⚠ 3B-Hallu" if r["3b_hallu_vs_gold"] else ""
        print(f"  {r['refnr']}: 7b={r['7b']!r:25s} 3b={r['3b']!r:25s}{gold_tag}{hallu_tag}")
    print()

### Drei 3B-Halluzinations-Klassen

| Klasse | 3B-Beispiele (refnr + Werte) | 7B sagt | Vermutete Ursache (Mechanismus) |
|---|---|---|---|
| **1. „Null wird zu 0" — Schema-Verletzung bei `gehalt_min_eur`** (74/89 Anzeigen!) | `10000-1202917922-S`: 3b=**0**, 7b=None / `10000-1203115555-S`: 3b=**0**, 7b=None / `10000-1203575012-S`: 3b=**0**, 7b=None | None (= im Text steht keine Zahl) | 3B unterscheidet **„kein Wert genannt"** nicht von **„0 Euro Gehalt"**. Schreibt 0 als Null-Surrogat. Ist explizite **Schema-Verletzung** (Schema sagt „ganze Zahl ODER null"); würde Aggregate ruinieren (Durchschnitt aller Gehälter stark verzerrt). Mechanismus: 3B fehlt das Konzept „fehlender Wert" — Default-Output ist eine Zahl, 0 ist der „neutralste" Default. |
| **2. Konsistenz-Regel verletzt — `gehalt_zeitraum` halluziniert** (22 Anzeigen) | `10000-1202201960-S`: 3b=`'jahr'`, 7b=None / `10000-1204807415-S`: 3b=`'jahr'`, 7b=None / `10001-1001641852-S`: 3b=`'jahr'`, 7b=None | None (passend zu None bei `gehalt_min_eur`) | Schema-Regel: *„wenn `gehalt_min_eur` null, MUSS `gehalt_zeitraum` null sein"*. 3B verletzt das systematisch und schreibt fast immer `'jahr'`. Mechanismus: 3B verarbeitet die 6 Felder **unabhängig**, ohne Kreuzregeln zwischen verwandten Feldern. 7B hält die Konsistenz; 3B fehlt die Kapazität für Multi-Hop-Reasoning über das Schema. |
| **3. Default-`mid`-Bias bei `erfahrungslevel`** (40 Anzeigen) | `10000-1203115555-S`: 3b=**mid**, 7b=junior (Ausbildung!) / `10000-1204371667-S`: 3b=**mid**, 7b=junior / `10000-1205331810-S`: 3b=**mid**, 7b=senior / `10000-1206279423-S`: 3b=**mid**, 7b=junior | klassengerechte Antwort (`junior`/`senior`/`egal`) | 3B fällt bei Unsicherheit auf **mid** zurück, statt das wahrscheinlichere Signal aus dem Text zu extrahieren (z.B. „Ausbildung zur Fachinformatikerin" → eindeutig junior). Klassischer **Mode-Collapse**: Modell wählt häufigste Klasse als Sicherheits-Default. Anders als bei Halluzination 1/2 ist das **kein Schema-Verstoß**, sondern nur niedrigere Accuracy — der Validator würde es nicht abfangen. |

**Bonus — `vertragsart`-Verwechslung** (15 Anzeigen, eigenständig erwähnenswert): 3B schreibt `werkstudent` bei `festanstellung`-Anzeigen oder umgekehrt. Beispiel: `10001-1002583827-S` (3b=`werkstudent`, 7b=`festanstellung`). Vermutlich nimmt 3B Erwähnungen von Werkstudenten/Praktikanten im Anzeigentext (oft im Boilerplate „wir bilden auch Werkstudenten aus") als Anstellungsart der ausgeschriebenen Stelle.

---

### Implikation — Würde ich 3B in Production einsetzen?

**Direkter Einsatz: Nein.** Die 74/89 `gehalt_min_eur`-Schema-Verstöße allein disqualifizieren 3B für strukturierte Datenextraktion — das sind 83% der Anzeigen mit einer harten Schema-Verletzung, die Aggregations-Statistiken systematisch verzerren würde. Plus die 22 Konsistenz-Verletzungen bei `gehalt_zeitraum`.

**Mit Sicherung — bedingt machbar:**
- **Pflicht-Validator vor Persistenz:** `python annotation/validate.py --validate-jsonl predictions_3b_full.jsonl` würde Halluzination 1+2 abfangen (Schema-Konsistenz: wenn `gehalt_min_eur` null, dann `gehalt_zeitraum` null). 3B-Outputs, die nicht durchgehen, gehen an 7B als Fallback.
- **Eingrenzung auf stabile Felder:** `vertragsart` (~83% in 3B, 15 Abweichungen von 89) ist tolerabel, wenn das Output-Feld unkritisch ist. `homeoffice` mit 18 Abweichungen ist im Rauschen verglichen mit 7B-Niveau.
- **Klassen-Default-Korrektur:** Halluzination 3 (Mid-Bias) ist statistisch korrigierbar — wenn ich weiß, dass 3B `mid` über-rapportiert, kann ich aus 3B-`mid`-Predictions Stichproben ziehen und manuell prüfen.

**Pro-3B-Argument:** ~3× schneller (~1.3s vs ~4s/Anzeige laut Cheatsheet), läuft auf 16-GB-V100 statt 32-GB-V100 (mehr Server-Auswahl: euler statt nur gauss). Bei einem **Pre-Filter-Setup** — „extrahiere grobe Klassifikation, dann teure Felder mit 7B nachschärfen" — könnte 3B die Pipeline um Faktor 2-3 beschleunigen.

**Pro-7B-Argument:** Strukturierte Extraktion ist eine **konsistenzkritische** Aufgabe. 7B verstand die Cross-Field-Regel (`null` bei min → `null` bei zeitraum) ohne explizite Erwähnung im Prompt; 3B nicht. Wenn Konsistenz wichtiger ist als Latenz, ist der 3× Speed-Win den Halluzinations-Aufwand nicht wert.

**→ Entscheidung für FIDP-Berufsalltag:** 7B als Default-Modell, 3B nur als **Fallback bei Server-Engpässen** und nur mit zwingendem Validator-Schritt. Für **mein konkretes Setup** (Skill-Aggregation aus Bundesagentur-Korpus) wäre 3B mit aktuellem Schema unbrauchbar, weil die Gehalts-Halluzinationen das Reporting kaputt machen würden.

## Phase 4 — Synthese

Iterations-Tabelle (Baseline / A / B mit Hypothese, Aktion, Δ Gesamt, Δ schwächstes Feld, Diagnose) + Synthese-Antworten zu den drei Fragen aus dem Aufgabenblatt.

## Phase 6 — Vollständiger 7B-Run + 3B-Halluzinations-Klassen

Per-Field-Accuracy auf den 12 Hand-Gold-Anzeigen + Schema-Konformitäts-Check auf den restlichen Anzeigen ohne Gold. 3B-vs-7B-vs-Gold per `refnr` joinen, drei eigenständige Halluzinations-Klassen mit konkreten Beispielen identifizieren.